In [7]:
# DIRECTOB2B/ETL/rutina_merge.py
import os
import sys
import time
import json
import pickle
import logging
import gc
import unicodedata
import re
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

# MANIPULACIÓN DE DATOS
import pandas as pd
import numpy as np

# BASE DE DATOS
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker, Session
from psycopg2.extras import execute_batch

# RED
import requests

In [8]:
# CONFIGURACIÓN
load_dotenv()

# LOGGING
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# --- CONFIGURACIÓN DB ---
def get_db_engine():
    try:
        user = os.getenv("DB_USER")
        password = os.getenv("DB_PASS")
        host = os.getenv("DB_HOST")
        port = os.getenv("DB_PORT")
        database = os.getenv("DB_NAME")
        
        if not all([user, password, host, port, database]):
            raise ValueError("Faltan variables de entorno para la BD")

        engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{database}')
        return engine
    except Exception as e:
        logger.error(f"Error conectando a BD: {e}")
        return None

In [ ]:
# --- UTILIDADES ---
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')
    text = re.sub(r'(\d+)\s*(ml|cm|mm|kg|g)', r'\1_\2', text)
    text = re.sub(r'[^a-z0-9\s\-_\/]', ' ', text)
    return text.strip()

In [9]:
# DEEP LEARNING / NLP
import tensorflow as tf                  # Framework de Deep Learning
from tensorflow.keras.utils import pad_sequences


In [10]:
engine = get_db_engine()

In [11]:
# Consulta para obtener productos y su subcategoría
query = """
        SELECT
        tp.csku,
        tp.cnombre,
        tp.cdescripcion,
        tp.cmarca,
        ts.nid as id_subcategoria,
        ts.cnombre_subcategoria AS subcategoria
        FROM
        tbl_producto AS tp
        LEFT JOIN tbl_subcategoria AS ts ON ts.nid = tp.nid_subcategoria;
        """

# Cargar resultados en DataFrame
df_productos = pd.read_sql(query, engine)
print("Catálogo obtenido correctamente")

Catálogo obtenido correctamente


In [13]:
engine.dispose()

In [16]:
BASE_DIR = os.path.abspath(os.getcwd())

In [20]:
# 2. Construir las rutas usando os.path.join (funciona en Windows y Linux)
path_modelo = os.path.join(BASE_DIR, "Red_neuronal", "modelo_categorias.keras")
path_tokenizer = os.path.join(BASE_DIR, "Red_neuronal", "tokenizer.pkl")
path_labelencoder = os.path.join(BASE_DIR, "Red_neuronal", "labelencoder.pkl")

In [21]:
model = tf.keras.models.load_model(path_modelo)

2026-03-11 13:44:42,083 - WARNING - TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.


In [22]:
# Carga del tokenizer
with open(path_tokenizer, "rb") as f:
    tokenizer = pickle.load(f)

# Carga del LabelEncoder
with open(path_labelencoder, "rb") as f:
    le = pickle.load(f)

c:\Users\MoisesEugenioNavaMar\Github\Directo_b2b_Exel_del_Norte\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [23]:
# Solo productos sin subcategoría asignada
df_productos = df_productos[df_productos["id_subcategoria"].isna()]

In [25]:
# Unir columnas relevantes para el modelo
df_productos["texto"] = (
    df_productos["cnombre"].fillna("").astype(str) + " " +
    df_productos["cdescripcion"].fillna("").astype(str) + " " +
        df_productos["cmarca"].fillna("").astype(str)
)


In [27]:
# Aplicar normalización
df_productos["texto_norm"] = df_productos["texto"].apply(normalize_text)

In [28]:
# Convertir texto a secuencias numéricas
secuencias = tokenizer.texts_to_sequences(df_productos["texto_norm"])

In [29]:
# Padding para igualar longitud de entrada del modelo
X = pad_sequences(secuencias, maxlen=200)

In [30]:
# Predicción (probabilidades por clase)
y_pred = model.predict(X, verbose=0)

In [31]:
# Obtener índice de la clase con mayor probabilidad
y_classes = np.argmax(y_pred, axis=1)

In [ ]:
# Convertir índice a etiqueta original
categorias_pred = le.inverse_transform(y_classes)

In [33]:
# Guardar resultado en el DataFrame
df_productos["categoria_predicha"] = categorias_pred

In [35]:
df_productos.drop(
    columns=[
        "cnombre",
        "cdescripcion",
        "cmarca",
        "id_subcategoria",
        "subcategoria",
        "texto",
        "texto_norm"
    ],
    inplace=True
 )


In [36]:
engine = get_db_engine()

In [37]:
try:
    query = """
        SELECT
            nid as id_subcategoria,
            cnombre_subcategoria AS subcategoria
        FROM
            tbl_subcategoria
        ORDER BY
            nid ASC;
                """

    df_subcategorias = pd.read_sql(query, engine)
    print("Subcategorias Obtenidas")

except Exception as e:
    print("Error al ejecutar la consulta:", e)

engine.dispose()

Subcategorias Obtenidas


In [39]:
# ============================================================
# MERGE Y ACTUALIZACIÓN EN BD
# ============================================================

df_productos = df_productos.merge(
    df_subcategorias,
    left_on="categoria_predicha",
    right_on="subcategoria",
    how="left"
)

In [40]:
df_productos.drop(
    columns=["categoria_predicha", "subcategoria"],
    inplace=True
)

In [42]:
 # Crear lista de tuplas (id_subcategoria, csku)
updates = list(zip(
    df_productos["id_subcategoria"],
    df_productos["csku"]
))

In [43]:
# Conexión cruda para psycopg2
conn = engine.raw_connection()
cursor = conn.cursor()

In [44]:
 # Consulta de actualización
query_update = """
    UPDATE tbl_producto
    SET nid_subcategoria = %s
    WHERE csku = %s;
"""

In [45]:
# Ejecución en lote (alto rendimiento)
execute_batch(cursor, query_update, updates)

In [46]:
# Confirmar cambios
conn.commit()

In [47]:
# ============================================================
# CIERRE DE CONEXIONES
# ============================================================

cursor.close()
conn.close()
engine.dispose()

In [48]:
# Eliminar objetos grandes
del model, tokenizer, le
del df_productos, df_subcategorias

In [49]:
# Limpiar sesión TensorFlow (MUY IMPORTANTE)
tf.keras.backend.clear_session()


2026-03-11 13:56:17,820 - WARNING - From c:\Users\MoisesEugenioNavaMar\Github\Directo_b2b_Exel_del_Norte\.venv\Lib\site-packages\keras\src\backend\common\global_state.py:82: The name tf.reset_default_graph is deprecated. Please use tf.compat.v1.reset_default_graph instead.



In [50]:
# Forzar garbage collector
gc.collect()

0

In [ ]:
def categorizador():
    # DEEP LEARNING / NLP
    import tensorflow as tf                  # Framework de Deep Learning
    from tensorflow.keras.utils import pad_sequences
    
    engine = get_db_engine()
    
    # Consulta para obtener productos y su subcategoría
    query = """
            SELECT
            tp.csku,
            tp.cnombre,
            tp.cdescripcion,
            tp.cmarca,
            ts.nid as id_subcategoria,
            ts.cnombre_subcategoria AS subcategoria
            FROM
            tbl_producto AS tp
            LEFT JOIN tbl_subcategoria AS ts ON ts.nid = tp.nid_subcategoria;
            """

    # Cargar resultados en DataFrame
    df_productos = pd.read_sql(query, engine)
    print("Catálogo obtenido correctamente")
    
    engine.dispose()
    
    BASE_DIR = os.path.abspath(os.getcwd())
    
    # 2. Construir las rutas usando os.path.join (funciona en Windows y Linux)
    path_modelo = os.path.join(BASE_DIR, "Red_neuronal", "modelo_categorias.keras")
    path_tokenizer = os.path.join(BASE_DIR, "Red_neuronal", "tokenizer.pkl")
    path_labelencoder = os.path.join(BASE_DIR, "Red_neuronal", "labelencoder.pkl")
    
    model = tf.keras.models.load_model(path_modelo)
    
    # Carga del tokenizer
    with open(path_tokenizer, "rb") as f:
        tokenizer = pickle.load(f)

    # Carga del LabelEncoder
    with open(path_labelencoder, "rb") as f:
        le = pickle.load(f)
        
    # Solo productos sin subcategoría asignada
    df_productos = df_productos[df_productos["id_subcategoria"].isna()]
    
    # Unir columnas relevantes para el modelo
    df_productos["texto"] = (
        df_productos["cnombre"].fillna("").astype(str) + " " +
        df_productos["cdescripcion"].fillna("").astype(str) + " " +
            df_productos["cmarca"].fillna("").astype(str)
    )
    
    # Aplicar normalización
    df_productos["texto_norm"] = df_productos["texto"].apply(normalize_text)
    
    # Convertir texto a secuencias numéricas
    secuencias = tokenizer.texts_to_sequences(df_productos["texto_norm"])
    
    # Padding para igualar longitud de entrada del modelo
    X = pad_sequences(secuencias, maxlen=300)
    
    # Obtener índice de la clase con mayor probabilidad
    y_classes = np.argmax(y_pred, axis=1)
    
    # Convertir índice a etiqueta original
    categorias_pred = le.inverse_transform(y_classes)
    
    # Guardar resultado en el DataFrame
    df_productos["categoria_predicha"] = categorias_pred
    
    df_productos.drop(
    columns=[
        "cnombre",
        "cdescripcion",
        "cmarca",
        "id_subcategoria",
        "subcategoria",
        "texto",
        "texto_norm"
    ],
    inplace=True
    )

    engine = get_db_engine()
    
    try:
    query = """
            SELECT
                nid as id_subcategoria,
                cnombre_subcategoria AS subcategoria
            FROM
                tbl_subcategoria
            ORDER BY
                nid ASC;
                    """

        df_subcategorias = pd.read_sql(query, engine)
        print("Subcategorias Obtenidas")

    except Exception as e:
        print("Error al ejecutar la consulta:", e)

    engine.dispose()
    
    # ============================================================
    # MERGE Y ACTUALIZACIÓN EN BD
    # ============================================================

    df_productos = df_productos.merge(
        df_subcategorias,
        left_on="categoria_predicha",
        right_on="subcategoria",
        how="left"
    )
    
    df_productos.drop(
    columns=["categoria_predicha", "subcategoria"],
    inplace=True
    )
    
     # Crear lista de tuplas (id_subcategoria, csku)
    updates = list(zip(
        df_productos["id_subcategoria"],
        df_productos["csku"]
    ))
    
    # Conexión cruda para psycopg2
    conn = engine.raw_connection()
    cursor = conn.cursor()
    
     # Consulta de actualización
    query_update = """
        UPDATE tbl_producto
        SET nid_subcategoria = %s
        WHERE csku = %s;
    """
    
    # Ejecución en lote (alto rendimiento)
    execute_batch(cursor, query_update, updates)
    
    # Confirmar cambios
    conn.commit()
    
    # ============================================================
    # CIERRE DE CONEXIONES
    # ============================================================

    cursor.close()
    conn.close()
    engine.dispose()
    
    # Eliminar objetos grandes
    del model, tokenizer, le
    del df_productos, df_subcategorias
    
    # Limpiar sesión TensorFlow (MUY IMPORTANTE)
    tf.keras.backend.clear_session()
    
    # Forzar garbage collector
    gc.collect()